# Conditional GAN extensions

Same comparison as [GAN_Tutorial.ipynb](../../GAN_Tutorial.ipynb): real samples beside generated samples, then the discriminator and generator losses. The tutorial draws 2D moons. These runs draw MNIST and Fashion-MNIST.

All six archives call one train function. The dataset changes in `01_get_data.load_data`. The discriminator experiment is `params["variant"]`.

- **cgan** — class-conditional MLP, binary cross-entropy, the tutorial update.
- **patchgan** — same generator, convolutional PatchGAN discriminator.
- **feature-matching-loss** — same MLP, generator loss adds an L1 match of discriminator features.

`final.png` uses the test-split labels, so it lines up with `real_test.png`. `classes.png` is one row per class. Within-class `diversity` and `sharpness` are the numeric companions to those grids. Feature-matching generator loss is not on the same scale as the other generator curves; the dashed BCE curve is the comparable adversarial term.

Run this notebook from the repository root after `scripts/mnist_gan_extensions/model_selection.py`.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return here


ROOT = project_root()
EXPERIMENTS = ("cgan", "patchgan", "feature-matching-loss")
DATASETS = ("digits", "fashion")


def archive(experiment: str, dataset: str) -> Path:
    return ROOT / "reports" / "mnist_gan_extensions" / "experiments" / experiment / dataset


def load_json(experiment: str, dataset: str, name: str) -> dict:
    path = archive(experiment, dataset) / name
    return json.loads(path.read_text(encoding="utf-8"))


histories = {
    (experiment, dataset): load_json(experiment, dataset, "history.json")
    for experiment in EXPERIMENTS
    for dataset in DATASETS
}
tests = {
    (experiment, dataset): load_json(experiment, dataset, "evaluation_test.json")
    for experiment in EXPERIMENTS
    for dataset in DATASETS
}
sources = {
    (experiment, dataset): load_json(experiment, dataset, "source.json")
    for experiment in EXPERIMENTS
    for dataset in DATASETS
}
rows = []
for experiment in EXPERIMENTS:
    for dataset in DATASETS:
        metrics = tests[(experiment, dataset)]
        source = sources[(experiment, dataset)]
        rows.append(
            {
                "experiment": experiment,
                "dataset": dataset,
                "epochs": source["epochs"],
                "diversity": metrics["diversity"],
                "class_separation": metrics["class_separation"],
                "sharpness": metrics["sharpness"],
                "real_sharpness": metrics["real_sharpness"],
                "sample_std": metrics["sample_std"],
                "real_std": metrics["real_std"],
                "mode_collapse": metrics["mode_collapse"],
                "generator_bce": metrics["generator_bce"],
            }
        )
summary = pd.DataFrame(rows)
summary


In [ ]:
fig, axes = plt.subplots(len(DATASETS), len(EXPERIMENTS), figsize=(12, 6), sharex=True)
for row, dataset in enumerate(DATASETS):
    for col, experiment in enumerate(EXPERIMENTS):
        axis = axes[row, col]
        history = histories[(experiment, dataset)]
        axis.plot(history["train_discriminator"], label="train D")
        axis.plot(history["train_generator_bce"], label="train G bce")
        axis.plot(history["eval_generator_bce"], label="eval G bce", linestyle="--")
        if experiment == "feature-matching-loss":
            axis.plot(history["train_generator"], label="train G objective", linestyle=":")
        axis.set_title(f"{dataset} / {experiment}")
        axis.set_xlabel("epoch")
        axis.set_ylabel("loss")
        axis.legend(fontsize=7)
fig.tight_layout()
plt.show()


## Samples

Each block is one dataset. The first row of images is the real test batch. The next rows are the class-matched generations (`final.png`) and the class grid (`classes.png`, ten rows).


In [ ]:
for dataset in DATASETS:
    display(Markdown(f"### {dataset}"))
    display(Markdown("Real test batch"))
    display(Image(filename=str(archive("cgan", dataset) / "real_test.png")))
    for experiment in EXPERIMENTS:
        metrics = tests[(experiment, dataset)]
        display(
            Markdown(
                f"**{experiment}** — diversity {metrics['diversity']:.3f}, "
                f"class separation {metrics['class_separation']:.3f}, "
                f"sharpness {metrics['sharpness']:.3f} "
                f"(real {metrics['real_sharpness']:.3f}), "
                f"mode collapse {metrics['mode_collapse']}"
            )
        )
        display(Image(filename=str(archive(experiment, dataset) / "final.png")))
        display(Image(filename=str(archive(experiment, dataset) / "classes.png")))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for axis, column, title in (
    (axes[0], "diversity", "within-class diversity"),
    (axes[1], "sharpness", "sharpness"),
):
    pivot = summary.pivot(index="experiment", columns="dataset", values=column)
    pivot = pivot.loc[list(EXPERIMENTS)]
    pivot.plot.bar(ax=axis, rot=15)
    axis.set_title(title)
    axis.set_xlabel("")
fig.tight_layout()
plt.show()


In [ ]:
lines = []
for dataset in DATASETS:
    block = summary[summary["dataset"] == dataset].set_index("experiment")
    base = block.loc["cgan"]
    for experiment in EXPERIMENTS[1:]:
        row = block.loc[experiment]
        diversity_delta = float(row["diversity"] - base["diversity"])
        sharpness_delta = float(row["sharpness"] - base["sharpness"])
        lines.append(
            f"{dataset}, {experiment} versus cgan: "
            f"diversity {diversity_delta:+.3f}, sharpness {sharpness_delta:+.3f}, "
            f"mode collapse {bool(row['mode_collapse'])}."
        )
display(Markdown("### Read of the numbers\n\n" + "\n".join(f"- {line}" for line in lines)))
